# Gold: embed the children

Task `gold_embed` of the job `ingest_financebench`, after `gold_chunk`. Gives every child in `gold_children` a vector
with `ai_query` on the embedding endpoint `databricks-gte-large-en` (1,024 numbers per child), and saves it in
`gold_embeddings`, **one row per child**: the child's columns, and its vector.

| Column | Type | Holds |
| --- | --- | --- |
| `child_id` | STRING | The child, such as `3M_2018_10K#42#text#0` (see `gold_children`) |
| `parent_id` | STRING | The parent section it belongs to |
| `doc_name` | STRING | The document name |
| `kind` | STRING | `text` or `table` |
| `text` | STRING | The child's text, exactly as in `gold_children` (`doc_name \| header`, a blank line, then the piece) |
| `page_start`, `page_end` | INT | The child's pages |
| `embedding` | ARRAY&lt;FLOAT&gt; | The vector of the child's text, 1,024 numbers |
| `embedded_at` | TIMESTAMP | When it was embedded |

**What is embedded.** The child's `text`, with a table's HTML tags taken out (`</tr>` becomes a line break, every other
tag a space) and cut at `MAX_EMBED_CHARS` characters, so a very large table stays inside the model's input limit. Only
the vector is made from this cleaned text. `text` itself is stored as it is, so the search returns the real child.

**Each child is paid for once.** `gold_chunk` replaces all the children of a document whenever a new part of it
arrives, so the vectors are not kept in `gold_children`: they would be lost and paid again. They live in their own table,
and a child is embedded again only when its text changed. A rerun with nothing new embeds nothing.

**Vector Search.** Change data feed is on, so a Delta Sync index with self-managed embeddings (`embedding`, dimension
1,024) can sync from `gold_embeddings`. Questions must be embedded with the same endpoint.

Written to [doc/code_standard.md](../../doc/code_standard.md).

In [ ]:
from pyspark.sql import functions as F

In [ ]:
CATALOG = "workspace"
GOLD_SCHEMA = f"{CATALOG}.finance_bench_gold"

CHILDREN_TABLE = f"{GOLD_SCHEMA}.gold_children"
EMBEDDINGS_TABLE = f"{GOLD_SCHEMA}.gold_embeddings"

ENDPOINT = "databricks-gte-large-en"  # 1,024 dimensions, input up to 8,192 tokens
MAX_EMBED_CHARS = 16000  # about 4,000 to 6,000 tokens, safely under the input limit even for HTML tables
DOCS_PER_BATCH = 20  # a save point: each batch of documents is written before the next is embedded

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {EMBEDDINGS_TABLE} (
  child_id STRING, parent_id STRING, doc_name STRING, kind STRING, text STRING, page_start INT, page_end INT,
  embedding ARRAY<FLOAT>, embedded_at TIMESTAMP
) TBLPROPERTIES (delta.enableChangeDataFeed = true)""")  # change data feed: a Vector Search index can sync from it

## Step 1: drop the vectors that no longer match a child

A vector is kept only while `gold_children` still has a child with the same `child_id` **and the same text**. A
`MERGE` with only `WHEN NOT MATCHED BY SOURCE THEN DELETE` deletes every other row of `gold_embeddings`: children that
are gone, and children whose text changed (they are embedded again in Step 2). It never inserts or changes a row.

**Example.** An earlier run chunked and embedded part 1 of `PGE_2020_10K`. Its last section, `PGE_2020_10K#57`
("Item 1A. Risk Factors"), runs on into part 2. Since then silver has saved part 2 of `PGE_2020_10K` and part 1 of
`AMD_2022_10K`, and `gold_chunk` has chunked both documents again.

**1a. Before.** Some of the rows of `PGE_2020_10K` in `gold_embeddings` (`text` shortened):

| child_id | text | embedding |
| --- | --- | --- |
| PGE_2020_10K#1#text#0 | PGE_2020_10K \| UNITED STATES⏎SECURITIES…⏎⏎Washington, D.C. 20549 | `[0.0213, -0.0087, …]` |
| PGE_2020_10K#3#table#1-8 | PGE_2020_10K \| (Mark One)⏎⏎`<table><tr><th>Commission File Number</th>…</table>` | `[-0.0154, 0.0302, …]` |
| PGE_2020_10K#57#text#0 | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎The Utility's operations… (1,900 characters) | `[0.0071, 0.0198, …]` |
| PGE_2020_10K#57#text#1 | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎…wildfires in its service area. (800 characters) | `[0.0119, -0.0240, …]` |

The same children in `gold_children` now. Section 57 grew with the text from part 2: piece 0 is the same, piece 1
is longer, and there is a new piece 2:

| child_id | text |
| --- | --- |
| PGE_2020_10K#1#text#0 | the same |
| PGE_2020_10K#3#table#1-8 | the same |
| PGE_2020_10K#57#text#0 | the same |
| PGE_2020_10K#57#text#1 | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎…wildfires in its service area. The Utility could… (1,950 characters) |
| PGE_2020_10K#57#text#2 | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎…and the cost of the Wildfire Fund… (700 characters) |

**1b. After `MERGE`.** `PGE_2020_10K#57#text#1` has no child with the same text, so its row is deleted. The other three
match a child exactly and stay, with their vectors: they are not paid again.

| child_id | embedding |
| --- | --- |
| PGE_2020_10K#1#text#0 | `[0.0213, -0.0087, …]` |
| PGE_2020_10K#3#table#1-8 | `[-0.0154, 0.0302, …]` |
| PGE_2020_10K#57#text#0 | `[0.0071, 0.0198, …]` |

In [ ]:
spark.sql(f"""
MERGE INTO {EMBEDDINGS_TABLE} AS e
USING {CHILDREN_TABLE} AS c
ON e.child_id = c.child_id AND e.text = c.text
WHEN NOT MATCHED BY SOURCE THEN DELETE""")

## Step 2: embed the children that have no vector

The children with no row in `gold_embeddings` (a `left_anti` join on `child_id`) are embedded with `ai_query` and
appended, `DOCS_PER_BATCH` documents at a time, so a failed run keeps the batches it has already written and the next
run continues from there. Only the document names are collected to the driver, not the data. The embedding runs on the
executors, many children at a time.

**Example.** Continues from Step 1.

**2a. After the `left_anti` join.** The children of `gold_children` with no vector: the changed piece, the new piece,
the sections of part 2 and all of `AMD_2022_10K`:

| child_id | parent_id | doc_name | kind | text | page_start | page_end |
| --- | --- | --- | --- | --- | --- | --- |
| PGE_2020_10K#57#text#1 | PGE_2020_10K#57 | PGE_2020_10K | text | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎…wildfires in its service area. The Utility could… | 31 | 52 |
| PGE_2020_10K#57#text#2 | PGE_2020_10K#57 | PGE_2020_10K | text | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎…and the cost of the Wildfire Fund… | 31 | 52 |
| PGE_2020_10K#58#text#0 | PGE_2020_10K#58 | PGE_2020_10K | text | PGE_2020_10K \| Item 1B. Unresolved Staff Comments⏎⏎None. | 53 | 53 |
| … | | | | | | |
| AMD_2022_10K#1#text#0 | AMD_2022_10K#1 | AMD_2022_10K | text | AMD_2022_10K \| UNITED STATES⏎SECURITIES AND EXCHANGE COMMISSION⏎⏎… | 1 | 1 |
| … | | | | | | |

**2b. `docs`.** `["AMD_2022_10K", "PGE_2020_10K"]`. Two documents, fewer than `DOCS_PER_BATCH`, so one batch.

**2c. The text sent to the model.** A text child is sent as it is. A table child loses its HTML tags:

| child_id | text | sent to `ai_query` |
| --- | --- | --- |
| PGE_2020_10K#58#text#0 | PGE_2020_10K \| Item 1B. Unresolved Staff Comments⏎⏎None. | the same |
| AMD_2022_10K#4#table#1-12 | AMD_2022_10K \| (Mark One)⏎⏎`<table><tr><th>Commission File Number</th><th>…</th></tr><tr><td>0-07882</td>…</table>` | AMD_2022_10K \| (Mark One)⏎⏎Commission File Number …⏎0-07882 … |

**2d. After `withColumn`, appended to `gold_embeddings`.** The children's columns as they are, the vector and the time:

| child_id | parent_id | doc_name | kind | text | page_start | page_end | embedding | embedded_at |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| PGE_2020_10K#57#text#1 | PGE_2020_10K#57 | PGE_2020_10K | text | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎…wildfires… | 31 | 52 | `[0.0102, -0.0231, …]` | 2026-11-25 10:04:12 |
| PGE_2020_10K#57#text#2 | PGE_2020_10K#57 | PGE_2020_10K | text | PGE_2020_10K \| Item 1A. Risk Factors⏎⏎…Wildfire Fund… | 31 | 52 | `[0.0047, 0.0166, …]` | 2026-11-25 10:04:12 |
| PGE_2020_10K#58#text#0 | PGE_2020_10K#58 | PGE_2020_10K | text | PGE_2020_10K \| Item 1B. Unresolved Staff Comments⏎⏎None. | 53 | 53 | `[-0.0310, 0.0088, …]` | 2026-11-25 10:04:12 |
| … | | | | | | | | |
| AMD_2022_10K#4#table#1-12 | AMD_2022_10K#4 | AMD_2022_10K | table | AMD_2022_10K \| (Mark One)⏎⏎`<table>…</table>` | 1 | 1 | `[0.0189, -0.0045, …]` | 2026-11-25 10:04:12 |
| … | | | | | | | | |

Every `embedding` has 1,024 numbers. `PGE_2020_10K` now has a vector for every child, once each. A rerun finds no
child without a vector, so `docs` is empty and nothing is embedded.

In [ ]:
missing = (
    spark.table(CHILDREN_TABLE)
    .join(spark.table(EMBEDDINGS_TABLE).select("child_id"), "child_id", "left_anti")
)
docs = sorted(row.doc_name for row in missing.select("doc_name").distinct().collect())  # a few names, not the data
print(f"{len(docs)} documents have children to embed")

for start in range(0, len(docs), DOCS_PER_BATCH):
    batch_docs = docs[start:start + DOCS_PER_BATCH]
    (
        missing.filter(F.col("doc_name").isin(batch_docs))
        .withColumn("embed_input", F.substring(  # a table's HTML tags out: a row per line, the other tags as spaces
            F.regexp_replace(F.regexp_replace("text", "</tr>", "\n"), "<[^>]+>", " "), 1, MAX_EMBED_CHARS))
        .withColumn("embedding", F.expr(f"ai_query('{ENDPOINT}', embed_input)"))
        .withColumn("embedded_at", F.current_timestamp())
        .drop("embed_input")
        .write.mode("append").saveAsTable(EMBEDDINGS_TABLE)
    )
    print(f"embedded documents {start + 1} to {start + len(batch_docs)} of {len(docs)}")

In [ ]:
display(spark.sql(f"""
SELECT (SELECT count(*) FROM {CHILDREN_TABLE}) AS children,
       (SELECT count(*) FROM {EMBEDDINGS_TABLE}) AS embedded,
       (SELECT count(DISTINCT doc_name) FROM {EMBEDDINGS_TABLE}) AS documents,
       (SELECT min(size(embedding)) FROM {EMBEDDINGS_TABLE}) AS min_dimension,
       (SELECT max(size(embedding)) FROM {EMBEDDINGS_TABLE}) AS max_dimension
"""))